In [1]:
%%capture
%xmode Minimal

import helpers.notebook_helpers

## How Python handles types

* In this example here, we have three variables of different data types: A string, a list, and a dictionary​
* Despite being of different types, the length function words on all three variables​
* In other words, a string, a list, and a dictionary all have items that can be counted, so the length function is like, _“Well, it looks like a collection or sequence and it acts like a collection or sequence, so it must be a collection or sequence, therefore I can count the number of items in it”​_


In [ ]:
example_1 = "Hello? World!"
print(f"`{example_1}` is of type {type(example_1)} and has the length {len(example_1)}")

example_2 = [1, 3, 5, 8]
print(f"`{example_2}` is of type {type(example_2)} and has the length {len(example_2)}")

example_3 = {"hello" : "world", "foo": "bar"}
print(f"`{example_3}` is of type {type(example_3)} and has the length {len(example_3)}")

## Problems with duck-typing
* If we call `bark` on a list that contains a Pug, a Labrador Retriever, and a Sphynx, we error out when attempting to call `bark` on a cat​
* Because Python is dynamically typed, this error isn’t caught until runtime whereas in statically typed languages, this likely wouldn’t even compile

In [ ]:
class Pug:
    def bark(self):
        print("The pug goes arf arf!")

class LabradorRetriever:
    def bark(self):
        print("The labrador retriever goes arf arf!")

class Wolfhound:
    def bark(self):
        print("The wolfhound goes arf arf!")

def make_dogs_bark(pets):
    for pet in pets:
        pet.bark()

pets = [Pug(), LabradorRetriever(), Wolfhound()]
make_dogs_bark(pets)

In [ ]:
class Sphynx:
    def meow(self):
        print("The sphynx goes meow meow!")

pets = [Pug(), LabradorRetriever(), Sphynx()]
make_dogs_bark(pets)

## Type annotations

One way to address some of the issues that arise with dynamic typing is with Python’s type annotations​

* Type annotations use code hints that specify the expected type for a variable, a class attribute, or a function parameter or return value​
* So, if we were to update our Pug class example, we could include an attribute like `name` and use type annotation to indicate that we expect the provided value to be a string​

In [ ]:
from typing import List

class Pug:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The pug {self.name} goes arf arf!")

class LabradorRetriever:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The labrador retriever {self.name} goes arf arf!")

class Wolfhound:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The wolfhound {self.name} goes arf arf!")


def make_dogs_bark(pets: List[Pug | LabradorRetriever | Wolfhound ]) -> None:
    for pet in pets:
        pet.bark() 

make_dogs_bark([Pug(name="Matty"), LabradorRetriever(name="Gary"), Wolfhound(name="Lily")])

**However**, it’s important to remember that while type annotations make the code more readable and maintainable, they are not enforced by the Python interpreter​
* So even if we use type hints to indicate that the list `pets` should include only breeds of dogs, there is nothing stopping this code from running and erroring out if we make a list that includes a cat​
* In order to check types, we’d need to use an additional tool like `mypy` in order to catch these errors, which is what we see in this example​

In [2]:
%%mypy --show-error-codes
from typing import List

class Pug:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The pug {self.name} goes arf arf!")

class LabradorRetriever:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The labrador retriever {self.name} goes arf arf!")

class Wolfhound:
    def __init__(self, name: str) -> None:
        self.name: str = name
    
    def bark(self) -> None:
        print(f"The wolfhound {self.name} goes arf arf!")

class Sphynx:
    def __init__(self, name: str) -> None:
        self.name: str = name

    def meow(self) -> None:
        print(f"The sphynx {self.name} goes meow meow!")

def make_dogs_bark(pets: List[Pug | LabradorRetriever | Wolfhound ]) -> None:
    for pet in pets:
        pet.bark() 

make_dogs_bark([Pug(name="Matty"), LabradorRetriever(name="Gary"), Sphynx(name="River")])

```text
mypy output: :35: error: List item 2 has incompatible type "Sphynx"; expected "Pug | LabradorRetriever | Wolfhound"  [list-item]
Found 1 error in 1 file (checked 1 source file)

```

## The manual validation tax
* So type annotations are helpful, but if we want to actually check types, we need additional tooling – so we’re still potentially stuck with runtime type errors​
* To solve that, we could write our own validators using functions​
* But this gets tedious fast and undermines the flexibility that duck-typing gives us and doesn’t take advantage of Python’s type hints​

In [ ]:
from typing import List

class Pug:
    def bark(self) -> None:
        print("The pug goes arf arf!")

class LabradorRetriever:
    def bark(self) -> None:
        print("The labrador retriever goes arf arf!")

class Wolfhound:
    def bark(self) -> None:
        print("The wolfhound goes arf arf!")

class Sphynx:
    def meow(self) -> None:
        print("The sphynx goes meow meow!")

def make_dogs_bark_safely(pets: List[Pug | LabradorRetriever | Wolfhound ]) -> None:
    """Manual validation in the function"""
    for pet in pets:
        if not hasattr(pet, 'bark') or not callable(getattr(pet, 'bark')):
            raise TypeError(f"Invalid dog type: {type(pet).__name__}")
        pet.bark()

# Now this fails immediately with a clear message:
pets = [Pug(), Sphynx()]
make_dogs_bark_safely(pets)

## Pydantic in practice: A Pug example
So let’s update our class of `Pug` using Pydantic​
* We inherit from `BaseModel` and use type annotations to create a schema with:​
  * Name: A string that must be at least one character long​
  * Age: An integer​
  * And a color: A string that defaults to “fawn”​
​

When we try to create a `Pug` with the wrong kind of data, Pydantic catches it immediately with a clear message that lets us know:​
* What the input should be​
* What was the input given​
* And what was the type of the input given​


In [ ]:
from pydantic import BaseModel, Field

class Pug(BaseModel):
    name: str = Field(..., min_length=1, description="The pug's name")
    age: int
    color: str = "fawn"
    
    def bark(self):
        print(f"The pug {self.name} goes arf arf!")

print("Creating a pug with name 123 and age '14' ...")
new_pug = Pug(name=123, age='14')

In [ ]:
print("Creating a pug with name 'Matty' and age '14' ...")

new_pug = Pug(name="Matty", age='14')

print(f"The new pug is named {new_pug.name} which is of type {type(new_pug.name)}")
print(f"The new pug is aged {new_pug.age} which is of type {type(new_pug.age)}")
print(f"The new pug has the color {new_pug.color} which is of type {type(new_pug.color)}")

## TODO

In [ ]:
from pydantic import BaseModel, Field
from pprint import pprint

class Pug(BaseModel):
    name: str = Field(..., min_length=2, description="The pug's name")
    age: int
    color: str = "fawn"
    
    def bark(self):
        print(f"The pug {self.name} goes arf arf!")

pprint(Pug.__pydantic_core_schema__)


{'cls': <class '__main__.Pug'>,
 'config': {'title': 'Pug'},
 'custom_init': False,
 'metadata': {'pydantic_js_functions': [<bound method BaseModel.__get_pydantic_json_schema__ of <class '__main__.Pug'>>]},
 'ref': '__main__.Pug:31910875152',
 'root_model': False,
 'schema': {'computed_fields': [],
            'fields': {'age': {'metadata': {},
                               'schema': {'type': 'int'},
                               'type': 'model-field'},
                       'color': {'metadata': {},
                                 'schema': {'default': 'fawn',
                                            'schema': {'type': 'str'},
                                            'type': 'default'},
                                 'type': 'model-field'},
                       'name': {'metadata': {'pydantic_js_updates': {'description': 'The '
                                                                                    "pug's "
                                                    

## TODO

In [9]:
print(type(Pug.__pydantic_validator__))
print(type(Pug.__pydantic_serializer__))

<class 'pydantic_core._pydantic_core.SchemaValidator'>
<class 'pydantic_core._pydantic_core.SchemaSerializer'>


## TODO

In [17]:
name = "Matty"
age = "14"

print(f"type of `name`: {type(name)}")
print(f"type of `age`: {type(age)}")

print("Creating a new Pug ...")

new_pug = Pug(name=name,age=age)
print(f"type of `name`: {type(new_pug.name)}")
print(f"type of `age`: {type(new_pug.age)}")

data = {"name": name, "age": age}
new_pug = Pug.__pydantic_validator__.validate_python(data)
print(new_pug)

type of `name`: <class 'str'>
type of `age`: <class 'str'>
Creating a new Pug ...
type of `name`: <class 'str'>
type of `age`: <class 'int'>
name='Matty' age=14 color='fawn'


## TODO

In [18]:
name = "J"
age = "fourteen"

malformed_pug = Pug(name=name, age=age)

ValidationError: 2 validation errors for Pug
name
  String should have at least 2 characters [type=string_too_short, input_value='J', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='fourteen', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [ ]:
from pydantic import ValidationError
from pprint import pprint

try:
    Pug(name=name, age=age)
except ValidationError as exc:
    pprint(exc.errors())

[{'type': 'string_too_short', 'loc': ('name',), 'msg': 'String should have at least 2 characters', 'input': 'J', 'ctx': {'min_length': 2}, 'url': 'https://errors.pydantic.dev/2.13/v/string_too_short'}, {'type': 'int_parsing', 'loc': ('age',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'fourteen', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}]
